# Lead Generation Pipeline — v1.0

**Owner:** Edo Sanjaya P (Jericho) · **Runtime:** Google Colab (free tier)

A six-stage pipeline that turns a raw company list into a scored, verified,
client-deliverable lead sheet.

| Stage | What it does | Output |
|---|---|---|
| 1. Ingest | Load raw leads from Sheets / CSV / Drive | `df_raw` |
| 2. Clean | Normalise, extract root domain, dedupe | `df_clean` |
| 3. Enrich | Fetch site, detect language + industry signals | `df_enriched` |
| 4. Verify | Email format + MX record check | `df_verified` |
| 5. Score | Transparent 100-point ICP rubric | `df_scored` |
| 6. Export | Multi-tab XLSX + optional Google Sheet | deliverable file |

**Design rules this notebook follows**
- Every knob lives in `CONFIG` (Cell 2). No magic numbers buried in functions.
- Scoring is *reason-before-score*: every point is traceable to a stated reason.
- Nothing is invented. Unknown fields stay `None`, never guessed.
- Network work checkpoints to Drive, so a Colab disconnect costs you minutes, not hours.
- `robots.txt` is checked before any page fetch.

**Run order:** Cell 1 → Cell 2 → ... top to bottom. Each stage is independently
re-runnable once the stage before it has produced its dataframe.


---
## Cell 1 — Install dependencies

Run once per Colab session (Colab wipes the VM when it disconnects).
Takes about 40 seconds.


In [ ]:
# --- Cell 1: dependencies -----------------------------------------------
%pip install -q tldextract dnspython openpyxl gspread-dataframe tqdm requests beautifulsoup4 lxml

print("Dependencies installed. Restart NOT required.")


---
## Cell 2 — SCENARIO + CONFIG

**Set `SCENARIO` on the first line, then edit only the run identity and input
block.** Everything else — ICP, weights, role keywords, tier cutoffs — switches
automatically.

| Scenario | You are looking for | What actually predicts a good lead |
|---|---|---|
| `client` | Companies who will buy from you | Budget authority + verified email |
| `member` | Organisations whose staff will join the gym | Headcount + physical proximity + HR contact |
| `supplier` | Companies you will buy from | Certifications + export capability + role |
| `distributor` | Companies who will resell for you | Channel reach + partnerships contact |

**Why the weights differ, not just the keywords.** For `client`, a verified email
is the deliverable, so it carries 30 points. For `supplier`, you are the buyer —
email is table stakes, and what matters is whether they can actually supply
(certifications, 25 points). For `member`, a lead 400km from Bandung is worth
zero regardless of how good it looks, so proximity carries real weight.

**Scoring dimensions.** Nine exist; each scenario uses a subset. A dimension
weighted `0` is skipped entirely and never appears in `score_reasons`.


In [ ]:
# --- Cell 2: SCENARIO + CONFIG -------------------------------------------

SCENARIO = "client"      # "client" | "member" | "supplier" | "distributor"


# =========================================================================
# SHARED TITLE VOCABULARY (composed into scenarios below)
# =========================================================================
CSUITE = [
    "founder", "co-founder", "cofounder",
    "ceo", "cto", "coo", "cmo", "cfo", "cro",
    "chief executive", "chief technology", "chief operating",
    "chief marketing", "chief financial", "chief revenue",
    "owner", "proprietor", "president", "director", "head of",
    "vp", "svp", "evp", "vice president", "partner",
    "managing director", "managing partner", "general manager", "principal",
]

# Chinese titles. Matched as substrings — CJK has no word boundaries.
# Longer titles listed before shorter ones they contain.
CSUITE_ZH = [
    "首席执行官", "执行长", "董事长", "董事總經理", "总经理", "總經理",
    "创始人", "創辦人", "创办人", "总裁", "總裁", "负责人", "負責人",
    "合伙人", "合夥人", "总监", "總監",
]

INFLUENCERS = [
    "manager", "lead", "supervisor", "specialist",
    "senior", "coordinator", "officer", "consultant",
    "经理", "經理", "主管", "主任", "专员", "專員",
]

EXCLUDED_ALWAYS = [
    "intern", "student", "trainee", "volunteer", "retired",
    "assistant to", "executive assistant", "seeking", "unemployed",
    "实习生", "學生", "学生",
]

APAC = ["singapore", "malaysia", "indonesia", "taiwan", "hong kong",
        "china", "thailand", "vietnam", "philippines", "japan", "korea"]
MANDARIN = ["china", "taiwan", "hong kong", "singapore", "macau"]


# =========================================================================
# BASE — identical across scenarios
# =========================================================================
BASE = {
    "client_name":   "Demo Client",
    "campaign_name": "run_01",
    "run_notes":     "First run.",

    # ---------- INPUT ----------
    "INPUT_MODE": "sample",            # sample | csv_upload | drive_csv | google_sheet
    "drive_csv_path":   "/content/drive/MyDrive/leadgen/raw_leads.csv",
    "google_sheet_id":  "",
    "google_sheet_tab": "Raw",

    "COLUMN_MAP": {
        "company_name": "Company",   "website":     "Website",
        "contact_name": "Contact Name", "job_title": "Title",
        "email":        "Email",     "country":     "Country",
        "city":         "City",      "employee_count": "Employees",
        "industry":     "Industry",  "linkedin_url": "LinkedIn",
    },

    # ---------- NETWORK ----------
    "REQUEST_DELAY_SEC": 2.0,
    "REQUEST_TIMEOUT":   10,
    "MAX_RETRIES":       2,
    "RESPECT_ROBOTS":    True,
    "USER_AGENT": "Mozilla/5.0 (compatible; LeadResearchBot/1.0; +research contact)",
    "ENRICH_LIMIT": None,

    # ---------- CHECKPOINTS / OUTPUT ----------
    "USE_DRIVE": False,
    "CHECKPOINT_DIR": "/content/drive/MyDrive/leadgen/checkpoints",
    "LOCAL_CHECKPOINT_DIR": "/content/checkpoints",
    "OUTPUT_DIR": "/content/output",
    "WRITE_TO_SHEET": False,
    "OUTPUT_SHEET_ID": "",
}


# =========================================================================
# SCENARIOS
# =========================================================================
SCENARIOS = {

# -------------------------------------------------------------------------
"client": {
    "label": "Potential clients (they buy from you)",
    "ICP": {
        "decision_maker_titles": CSUITE + CSUITE_ZH,
        "influencer_titles":     INFLUENCERS,
        "excluded_titles":       EXCLUDED_ALWAYS,
        "employee_min": 10, "employee_max": 500,
        # What business are they in?
        "sector_keywords": ["saas", "software", "platform", "b2b", "logistics",
                            "manufacturing", "export", "wholesale", "distribution",
                            "agency", "consulting"],
        # What role do they play in the chain? Not used for clients.
        "role_keywords": [],
        # Proof of capability. Not used for clients.
        "credential_keywords": [],
        "target_countries": APAC, "mandarin_markets": MANDARIN,
        "target_cities": [],
    },
    "WEIGHTS": {
        "email_verified": 30, "decision_maker": 20, "company_size_fit": 15,
        "sector_match": 15, "role_match": 0, "credential_match": 0,
        "apac_mandarin_edge": 10, "local_proximity": 0, "website_live": 10,
    },
    "TIERS": {"A": 75, "B": 55, "C": 35},
    "HARD_GATES": {},
},

# -------------------------------------------------------------------------
"member": {
    "label": "Corporate / institutional gym accounts (Osbond)",
    # NOTE: this scores ORGANISATIONS whose staff become members —
    # corporates, hospitals, universities, banks. It does not score
    # individual walk-in prospects; those need a different data source.
    "ICP": {
        # HR and people functions hold the wellness budget, not the CEO.
        "decision_maker_titles": [
            "hr director", "head of hr", "hr manager", "human resources",
            "people operations", "head of people", "chief people",
            "benefits", "total rewards", "compensation and benefits",
            "general affairs", "office manager", "admin manager",
            "wellness", "occupational health",
            "hrd", "kepala hrd", "manajer sdm", "人力资源", "人事经理",
        ] + ["ceo", "founder", "owner", "general manager", "president"],
        "influencer_titles": INFLUENCERS,
        "excluded_titles":   EXCLUDED_ALWAYS,
        # Bigger is better here: headcount is the revenue driver.
        "employee_min": 50, "employee_max": 20000,
        "sector_keywords": ["bank", "hospital", "clinic", "university",
                            "college", "insurance", "corporate", "office",
                            "technology", "manufacturing", "government",
                            "rumah sakit", "universitas", "perusahaan"],
        "role_keywords": [],
        # Signals they already spend on staff wellbeing.
        "credential_keywords": ["employee benefits", "wellness program",
                                "corporate wellness", "staff benefits",
                                "kesejahteraan karyawan", "tunjangan"],
        "target_countries": ["indonesia"],
        "mandarin_markets": [],
        # Proximity is decisive. Nearest first.
        "target_cities": ["bandung", "cimahi", "padalarang", "soreang",
                          "lembang", "jatinangor", "west java", "jawa barat"],
    },
    "WEIGHTS": {
        "email_verified": 20, "decision_maker": 25, "company_size_fit": 25,
        "sector_match": 10, "role_match": 0, "credential_match": 0,
        "apac_mandarin_edge": 0, "local_proximity": 15, "website_live": 5,
    },
    "TIERS": {"A": 70, "B": 50, "C": 30},
    # A gym lead outside the service area is worth zero no matter how good it
    # looks on every other dimension. Weighting cannot express that; a gate can.
    "HARD_GATES": {"require_service_area": True},
},

# -------------------------------------------------------------------------
"supplier": {
    "label": "Potential suppliers (you buy from them)",
    "ICP": {
        # You want sales-side contacts — they are paid to reply to buyers.
        "decision_maker_titles": [
            "sales director", "sales manager", "export manager",
            "international sales", "business development", "account manager",
            "commercial director", "head of sales", "trade manager",
            "外贸经理", "销售经理", "出口经理",
        ] + CSUITE + CSUITE_ZH,
        "influencer_titles": INFLUENCERS,
        "excluded_titles":   EXCLUDED_ALWAYS,
        "employee_min": 20, "employee_max": 10000,
        "sector_keywords": ["manufacturer", "factory", "produce", "packaging",
                            "ingredients", "raw material", "processing",
                            "cold chain", "agriculture", "beverage", "food"],
        # THE key question for suppliers: what role in the chain?
        "role_keywords": ["manufacturer", "oem", "odm", "factory direct",
                          "supplier", "producer", "mill", "refinery",
                          "grower", "farm", "processor",
                          "生产厂家", "制造商", "工厂", "厂家", "pabrik", "produsen"],
        # Proof they can actually deliver. Heaviest weight in this scenario.
        "credential_keywords": ["iso 9001", "iso 22000", "haccp", "brc",
                                "fssc", "gmp", "halal", "kosher", "organic",
                                "fda registered", "ce certified", "sgs",
                                "bpom", "sni", "moq", "export license",
                                "years of experience", "since 19", "since 20"],
        "target_countries": APAC, "mandarin_markets": MANDARIN,
        "target_cities": [],
    },
    "WEIGHTS": {
        "email_verified": 15, "decision_maker": 10, "company_size_fit": 5,
        "sector_match": 10, "role_match": 20, "credential_match": 25,
        "apac_mandarin_edge": 10, "local_proximity": 0, "website_live": 5,
    },
    "TIERS": {"A": 70, "B": 50, "C": 30},
    "HARD_GATES": {},
},

# -------------------------------------------------------------------------
"distributor": {
    "label": "Potential distributors / resellers (they sell for you)",
    "ICP": {
        # A distributor's CEO will not answer. Their channel/BD lead will.
        "decision_maker_titles": [
            "business development", "channel", "partnerships", "partner manager",
            "sales director", "head of sales", "commercial director",
            "country manager", "regional manager", "distribution manager",
            "buying director", "category manager", "procurement",
            "manajer penjualan", "kepala cabang", "渠道经理", "招商经理",
        ] + ["ceo", "founder", "owner", "managing director", "general manager"],
        "influencer_titles": INFLUENCERS,
        "excluded_titles":   EXCLUDED_ALWAYS,
        # Reach is the product, so bigger is better.
        "employee_min": 20, "employee_max": 5000,
        "sector_keywords": ["fmcg", "retail", "horeca", "hospitality",
                            "catering", "food service", "beverage",
                            "supermarket", "convenience", "e-commerce"],
        "role_keywords": ["distributor", "authorized dealer", "authorised dealer",
                          "reseller", "wholesaler", "wholesale", "trading company",
                          "importer", "agent", "stockist", "channel partner",
                          "代理商", "经销商", "批发", "distributor resmi",
                          "agen resmi", "grosir", "penyalur"],
        "credential_keywords": ["nationwide", "warehouse", "fleet", "cold storage",
                                "outlets", "branches", "coverage", "logistics network",
                                "jaringan", "cabang", "gudang"],
        "target_countries": ["indonesia"] + APAC, "mandarin_markets": MANDARIN,
        "target_cities": [],
    },
    "WEIGHTS": {
        "email_verified": 20, "decision_maker": 20, "company_size_fit": 20,
        "sector_match": 5, "role_match": 20, "credential_match": 5,
        "apac_mandarin_edge": 5, "local_proximity": 0, "website_live": 5,
    },
    "TIERS": {"A": 70, "B": 50, "C": 30},
    "HARD_GATES": {},
},
}


# =========================================================================
# ASSEMBLE + VALIDATE
# =========================================================================
assert SCENARIO in SCENARIOS, f"Unknown SCENARIO '{SCENARIO}'. Pick: {list(SCENARIOS)}"

CONFIG = {**BASE, **SCENARIOS[SCENARIO], "scenario": SCENARIO}

DIMENSIONS = ["email_verified", "decision_maker", "company_size_fit",
              "sector_match", "role_match", "credential_match",
              "apac_mandarin_edge", "local_proximity", "website_live"]

# Every scenario must be internally coherent — fail now, not three stages later.
for name, sc in SCENARIOS.items():
    total = sum(sc["WEIGHTS"].values())
    assert total == 100, f"'{name}' weights total {total}, must be 100"
    assert set(sc["WEIGHTS"]) == set(DIMENSIONS), f"'{name}' has wrong dimension keys"
    t = sc["TIERS"]
    assert t["A"] > t["B"] > t["C"], f"'{name}' tier cutoffs must descend"
    # A dimension with weight > 0 needs the keywords to actually score it.
    if sc["WEIGHTS"]["role_match"] > 0:
        assert sc["ICP"]["role_keywords"], f"'{name}' weights role_match but has no role_keywords"
    if sc["WEIGHTS"]["credential_match"] > 0:
        assert sc["ICP"]["credential_keywords"], f"'{name}' weights credential_match but has none"
    assert "HARD_GATES" in sc, f"'{name}' missing HARD_GATES (use {{}} if none)"
    if sc["HARD_GATES"].get("require_service_area"):
        assert sc["ICP"]["target_cities"], f"'{name}' gates on service area but has no target_cities"
    if sc["WEIGHTS"]["local_proximity"] > 0:
        assert sc["ICP"]["target_cities"], f"'{name}' weights local_proximity but has no target_cities"

ACTIVE = {d: w for d, w in CONFIG["WEIGHTS"].items() if w > 0}

print(f"SCENARIO: {SCENARIO} — {CONFIG['label']}")
print(f"Client: {CONFIG['client_name']} | Campaign: {CONFIG['campaign_name']}")
print(f"Employee band: {CONFIG['ICP']['employee_min']}-{CONFIG['ICP']['employee_max']}")
print(f"Tiers: A>={CONFIG['TIERS']['A']} B>={CONFIG['TIERS']['B']} C>={CONFIG['TIERS']['C']}")
print("\nActive scoring dimensions:")
for d, w in sorted(ACTIVE.items(), key=lambda x: -x[1]):
    print(f"   {w:>3} pts  {d}")
print(f"   {sum(ACTIVE.values()):>3} pts  TOTAL")
if CONFIG["HARD_GATES"]:
    print(f"\nHard gates active: {list(CONFIG['HARD_GATES'])}")
    print("   Leads failing a gate are marked tier X and excluded from delivery.")
print("\nAll 4 scenarios validated.")


---
## Cell 3 — Imports and helpers

Sets up logging, the checkpoint system, and the shared HTTP session. Nothing
client-specific here — you should never need to edit this cell.


In [ ]:
# --- Cell 3: imports + shared helpers ------------------------------------
import os, re, json, time, socket, logging, warnings
from datetime import datetime, timezone
from urllib.parse import urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
import tldextract
import dns.resolver
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("leadgen")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log.info(f"Run id: {RUN_ID}")


# ---------- checkpointing ----------
def _ckpt_dir():
    d = CONFIG["CHECKPOINT_DIR"] if CONFIG["USE_DRIVE"] else CONFIG["LOCAL_CHECKPOINT_DIR"]
    os.makedirs(d, exist_ok=True)
    return d

def save_checkpoint(df, name):
    # Persist a stage output so a Colab disconnect does not cost you the work.
    path = os.path.join(_ckpt_dir(), f"{CONFIG['campaign_name']}__{name}.parquet")
    df.to_parquet(path, index=False)
    log.info(f"Checkpoint saved: {path} ({len(df)} rows)")
    return path

def load_checkpoint(name):
    path = os.path.join(_ckpt_dir(), f"{CONFIG['campaign_name']}__{name}.parquet")
    if os.path.exists(path):
        df = pd.read_parquet(path)
        log.info(f"Checkpoint loaded: {name} ({len(df)} rows)")
        return df
    log.info(f"No checkpoint found for '{name}'")
    return None


# ---------- HTTP ----------
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": CONFIG["USER_AGENT"],
    "Accept-Language": "en,zh;q=0.8,id;q=0.7",
})

_ROBOTS_CACHE = {}

def robots_allows(url):
    # Check robots.txt once per domain, then cache the verdict.
    if not CONFIG["RESPECT_ROBOTS"]:
        return True
    try:
        p = urlparse(url)
        base = f"{p.scheme}://{p.netloc}"
        if base not in _ROBOTS_CACHE:
            rp = RobotFileParser()
            rp.set_url(base + "/robots.txt")
            try:
                rp.read()
                _ROBOTS_CACHE[base] = rp
            except Exception:
                _ROBOTS_CACHE[base] = None      # unreachable robots -> allow
        rp = _ROBOTS_CACHE[base]
        if rp is None:
            return True
        return rp.can_fetch(CONFIG["USER_AGENT"], url)
    except Exception:
        return True

def fetch(url):
    # Polite GET with retries. Returns (html, status_note).
    if not robots_allows(url):
        return None, "blocked_by_robots"
    last = "unknown_error"
    for attempt in range(CONFIG["MAX_RETRIES"] + 1):
        try:
            r = SESSION.get(url, timeout=CONFIG["REQUEST_TIMEOUT"], allow_redirects=True)
            if r.status_code == 200:
                return r.text, "ok"
            last = f"http_{r.status_code}"
            if 400 <= r.status_code < 500:
                break                     # client errors will not fix themselves
        except requests.exceptions.SSLError:
            last = "ssl_error"; break
        except requests.exceptions.ConnectTimeout:
            last = "timeout"
        except requests.exceptions.ConnectionError:
            last = "connection_error"
        except Exception as e:
            last = f"error_{type(e).__name__}"
        time.sleep(1.5 * (attempt + 1))
    return None, last

print("Helpers ready.")


---
## Cell 4 — Stage 1: Ingest

Loads raw leads and applies `COLUMN_MAP`. If `INPUT_MODE` is `"sample"` this
generates a small realistic dataset so you can run the whole notebook end to end
before touching client data — including deliberately broken rows, so you can see
how the pipeline handles them.


In [ ]:
# --- Cell 4: Stage 1 — Ingest --------------------------------------------
PIPELINE_COLS = list(CONFIG["COLUMN_MAP"].keys())

def _sample_data():
    # Deliberately messy: duplicates, bad emails, dead domains, missing fields.
    rows = [
        ("Rui Feng Logistics", "https://www.ruifeng-logistics.com", "Chen Wei", "Chief Executive Officer", "chen.wei@ruifeng-logistics.com", "Taiwan", "Taipei", 120, "Logistics", ""),
        ("Rui Feng Logistics Co Ltd", "http://ruifeng-logistics.com/", "Chen Wei", "CEO", "chen.wei@ruifeng-logistics.com", "Taiwan", "Taipei", 120, "Logistics", ""),
        ("Meridian SaaS Pte Ltd", "https://meridian-saas.example.sg", "Aisyah Rahman", "Head of Growth", "aisyah@meridian-saas.example.sg", "Singapore", "Singapore", 45, "Software", ""),
        ("Nusantara Export", "https://nusantaraexport.co.id", "Budi Santoso", "Owner", "budi@nusantaraexport.co.id", "Indonesia", "Surabaya", 30, "Export", ""),
        ("Bright Path Interns", "https://brightpath.example.com", "Sam Lee", "Marketing Intern", "sam@brightpath.example.com", "Malaysia", "Penang", 8, "Agency", ""),
        ("Global Mega Corp", "https://globalmega.example.com", "Jane Doe", "Director of Operations", "jane.doe@globalmega.example.com", "United States", "Austin", 12000, "Manufacturing", ""),
        ("Hong Kong Trade Partners", "https://hktradepartners.example.hk", "Lam Ka Yiu", "Managing Director", "lam@hktradepartners.example.hk", "Hong Kong", "Kowloon", 65, "Distribution", ""),
        ("Broken Email Ltd", "https://broken-email.example.com", "No Name", "COO", "not-an-email", "Vietnam", "Hanoi", 90, "Software", ""),
        ("No Website Sdn Bhd", "", "Tan Mei Ling", "General Manager", "tan@nowebsite.example.my", "Malaysia", "Johor", 55, "Wholesale", ""),
        ("Dead Domain Industries", "https://this-domain-does-not-resolve-xyz123.com", "Ghost User", "CTO", "ghost@this-domain-does-not-resolve-xyz123.com", "Thailand", "Bangkok", 200, "Manufacturing", ""),
        ("上海精密制造有限公司", "https://shanghai-precision.example.cn", "Zhang Wei", "总经理", "zhang.wei@shanghai-precision.example.cn", "China", "Shanghai", 180, "Manufacturing", ""),
        ("Taipei Software Group", "https://taipei-soft.example.tw", "Lin Yi Chen", "C.E.O.", "lin@taipei-soft.example.tw", "Taiwan", "Taipei", 75, "Software", ""),
        # Bandung-area rows: only these can qualify under the 'member' gate.
        ("Bank Nusantara Bandung", "https://banknusantara.example.co.id", "Rina Wijaya", "HR Director", "rina.wijaya@banknusantara.example.co.id", "Indonesia", "Bandung", 850, "Bank", ""),
        ("Rumah Sakit Harapan Sehat", "https://rsharapansehat.example.co.id", "Dr. Andi Kurniawan", "Kepala HRD", "andi@rsharapansehat.example.co.id", "Indonesia", "Bandung", 420, "Rumah Sakit", ""),
        ("Universitas Cimahi Mandiri", "https://ucm.example.ac.id", "Sri Lestari", "Manajer SDM", "sri.lestari@ucm.example.ac.id", "Indonesia", "Cimahi", 310, "Universitas", ""),
        ("Warung Kecil Bandung", "https://warungkecil.example.co.id", "Asep Sutisna", "Owner", "asep@warungkecil.example.co.id", "Indonesia", "Bandung", 6, "Food", ""),
    ]
    cols = ["company_name","website","contact_name","job_title","email",
            "country","city","employee_count","industry","linkedin_url"]
    return pd.DataFrame(rows, columns=cols)


def ingest():
    mode = CONFIG["INPUT_MODE"]

    if mode == "sample":
        df = _sample_data()
        log.info(f"Sample data loaded: {len(df)} rows (includes intentional bad rows)")
        return df

    if mode == "csv_upload":
        from google.colab import files
        up = files.upload()                       # opens a file picker
        fname = list(up.keys())[0]
        df = pd.read_csv(fname)

    elif mode == "drive_csv":
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        df = pd.read_csv(CONFIG["drive_csv_path"])

    elif mode == "google_sheet":
        from google.colab import auth
        import gspread
        from google.auth import default
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        ws = gc.open_by_key(CONFIG["google_sheet_id"]).worksheet(CONFIG["google_sheet_tab"])
        df = pd.DataFrame(ws.get_all_records())

    # ---- apply COLUMN_MAP ----
    out = pd.DataFrame()
    missing = []
    for pipeline_col, source_col in CONFIG["COLUMN_MAP"].items():
        if source_col and source_col in df.columns:
            out[pipeline_col] = df[source_col]
        else:
            out[pipeline_col] = None
            if source_col:
                missing.append(f"{pipeline_col} <- '{source_col}'")
    if missing:
        log.warning("Columns not found in source, filled with None: " + ", ".join(missing))
    log.info(f"Ingested {len(out)} rows from {mode}")
    return out


df_raw = ingest()
df_raw["_source_row"] = range(len(df_raw))
save_checkpoint(df_raw, "01_raw")
display(df_raw.head(10))
print(f"\nShape: {df_raw.shape}")


---
## Cell 5 — Stage 2: Clean

The important move here is **deduping on root domain, not company name**.
"Rui Feng Logistics" and "Rui Feng Logistics Co Ltd" are the same company; no
string-similarity trick catches every variant of that, but both resolve to
`ruifeng-logistics.com`. Domain is the reliable key.


In [ ]:
# --- Cell 5: Stage 2 — Clean ---------------------------------------------
def norm_text(v):
    if pd.isna(v) or v is None:
        return None
    s = str(v).strip()
    s = re.sub(r"\s+", " ", s)
    return s or None

def norm_url(v):
    # Return a canonical https URL, or None if unusable.
    s = norm_text(v)
    if not s:
        return None
    if not s.startswith(("http://", "https://")):
        s = "https://" + s
    try:
        p = urlparse(s)
        if not p.netloc:
            return None
        netloc = p.netloc.lower().replace("www.", "")
        return urlunparse(("https", netloc, p.path.rstrip("/"), "", "", ""))
    except Exception:
        return None

def root_domain(v):
    # ruifeng-logistics.com from any URL or email
    s = norm_text(v)
    if not s:
        return None
    if "@" in s:
        s = s.split("@")[-1]
    ext = tldextract.extract(s)
    if not ext.domain or not ext.suffix:
        return None
    return f"{ext.domain}.{ext.suffix}".lower()

def norm_employees(v):
    # Handles 120, "120", "51-200", "1,200+", "" -> int or None
    if pd.isna(v) or v is None or str(v).strip() == "":
        return None
    s = str(v).replace(",", "")
    nums = re.findall(r"\d+", s)
    if not nums:
        return None
    nums = [int(n) for n in nums]
    return int(sum(nums) / len(nums)) if len(nums) > 1 else nums[0]


def clean(df):
    d = df.copy()

    for c in ["company_name", "contact_name", "job_title", "country", "city", "industry"]:
        d[c] = d[c].apply(norm_text)

    d["country"] = d["country"].apply(lambda x: x.lower() if x else None)
    d["email"] = d["email"].apply(lambda x: norm_text(x).lower() if norm_text(x) else None)
    d["website"] = d["website"].apply(norm_url)
    d["linkedin_url"] = d["linkedin_url"].apply(norm_url)
    d["employee_count"] = d["employee_count"].apply(norm_employees)

    # Domain key: prefer website, fall back to email domain.
    d["domain"] = d["website"].apply(root_domain)
    d.loc[d["domain"].isna(), "domain"] = d.loc[d["domain"].isna(), "email"].apply(root_domain)

    before = len(d)

    # Rows with neither a domain nor an email cannot be delivered. Park them.
    unusable = d[d["domain"].isna() & d["email"].isna()].copy()
    unusable["_drop_reason"] = "no domain and no email"
    d = d[~(d["domain"].isna() & d["email"].isna())].copy()

    # Dedupe: one row per (domain, email). Keep the row with the most filled fields.
    d["_completeness"] = d.notna().sum(axis=1)
    d = (d.sort_values("_completeness", ascending=False)
           .drop_duplicates(subset=["domain", "email"], keep="first")
           .drop(columns=["_completeness"])
           .sort_values("_source_row")
           .reset_index(drop=True))

    log.info(f"Clean: {before} -> {len(d)} rows "
             f"({before - len(d)} removed: {len(unusable)} unusable, "
             f"{before - len(d) - len(unusable)} duplicates)")
    return d, unusable


df_clean, df_dropped = clean(df_raw)
save_checkpoint(df_clean, "02_clean")

print("\nDropped rows:")
display(df_dropped[["company_name", "_drop_reason"]] if len(df_dropped) else "none")
display(df_clean[["company_name", "domain", "email", "job_title", "employee_count", "country"]])


---
## Cell 6 — Stage 3: Enrich

The slow stage: one HTTP request per company, throttled to one every
`REQUEST_DELAY_SEC`. At the default 2s, 100 leads takes about 4 minutes.

Per company it fetches the homepage and scans the text for four separate
keyword families, kept apart on purpose:

- **sector** — what industry they are in
- **role** — where they sit in the chain (manufacturer vs distributor vs retailer)
- **credential** — proof of capability (ISO, HACCP, halal, fleet size, MOQ)
- **locality** — whether they mention a target city

Sector and role must stay separate. A juice manufacturer and its distributor
share every sector keyword; only the role keywords tell them apart.

Language detection uses CJK character ratio — no extra dependency, and steadier
than statistical detectors on short marketing copy.

Checkpoints every 25 rows. If Colab disconnects, re-run this cell and it resumes
rather than re-fetching.


In [ ]:
# --- Cell 6: Stage 3 — Enrich --------------------------------------------
CJK_RE   = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]")
EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")

def detect_language(text):
    if not text or len(text) < 40:
        return None
    sample = text[:5000]
    cjk = len(CJK_RE.findall(sample))
    letters = sum(c.isalpha() for c in sample)
    if letters == 0:
        return None
    if cjk / max(letters, 1) > 0.15:
        return "zh"
    ascii_letters = sum(c.isascii() and c.isalpha() for c in sample)
    return "en" if ascii_letters / letters > 0.85 else "other"


def find_keywords(text_lower, keywords):
    # Returns the keywords present. Latin terms use word boundaries so
    # 'agent' does not fire inside 'agentic'; CJK uses substring.
    hits = []
    for kw in keywords:
        if CJK_RE.search(kw):
            if kw in text_lower:
                hits.append(kw)
        elif re.search(r"\b" + re.escape(kw) + r"\b", text_lower):
            hits.append(kw)
    return hits


KEYWORD_FAMILIES = {
    "sector":     "sector_keywords",
    "role":       "role_keywords",
    "credential": "credential_keywords",
    "locality":   "target_cities",
}

def enrich_one(row):
    out = {"site_status": None, "site_title": None, "site_language": None,
           "discovered_emails": None, "site_text_chars": 0}
    for fam in KEYWORD_FAMILIES:
        out[f"{fam}_matches"] = None
        out[f"{fam}_match_count"] = 0

    url = row.get("website")
    if not url:
        out["site_status"] = "no_website"
        return out

    html, status = fetch(url)
    out["site_status"] = status
    if html is None:
        return out

    try:
        soup = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        text = soup.get_text(" ", strip=True)

        out["site_title"]      = (soup.title.string or "").strip()[:200] if soup.title else None
        out["site_text_chars"] = len(text)
        out["site_language"]   = detect_language(text)

        low = text.lower()
        for fam, cfg_key in KEYWORD_FAMILIES.items():
            hits = find_keywords(low, CONFIG["ICP"].get(cfg_key, []))
            out[f"{fam}_matches"]     = ", ".join(hits[:8]) if hits else None
            out[f"{fam}_match_count"] = len(hits)

        found = {e.lower() for e in EMAIL_RE.findall(html)}
        found = {e for e in found
                 if not e.endswith((".png", ".jpg", ".gif", ".svg", ".webp"))}
        out["discovered_emails"] = ", ".join(sorted(found)[:5]) if found else None
    except Exception as e:
        out["site_status"] = f"parse_error_{type(e).__name__}"
    return out


ENRICH_COLS = (["site_status", "site_title", "site_language",
                "discovered_emails", "site_text_chars"]
               + [f"{f}_matches" for f in KEYWORD_FAMILIES]
               + [f"{f}_match_count" for f in KEYWORD_FAMILIES])

def enrich(df):
    d = df.copy()

    partial = load_checkpoint("03_enriched_partial")
    if partial is not None and set(ENRICH_COLS).issubset(partial.columns):
        d = d.merge(partial[["_source_row"] + ENRICH_COLS], on="_source_row", how="left")
    else:
        for c in ENRICH_COLS:
            d[c] = None

    todo = d[d["site_status"].isna()].index.tolist()
    if CONFIG["ENRICH_LIMIT"]:
        todo = todo[:CONFIG["ENRICH_LIMIT"]]

    log.info(f"Enriching {len(todo)} of {len(d)} rows "
             f"(~{len(todo) * CONFIG['REQUEST_DELAY_SEC'] / 60:.1f} min)")

    for n, idx in enumerate(tqdm(todo, desc="Enriching")):
        for k, v in enrich_one(d.loc[idx].to_dict()).items():
            d.at[idx, k] = v
        time.sleep(CONFIG["REQUEST_DELAY_SEC"])
        if (n + 1) % 25 == 0:
            save_checkpoint(d, "03_enriched_partial")

    save_checkpoint(d, "03_enriched_partial")
    return d


df_enriched = enrich(df_clean)
save_checkpoint(df_enriched, "03_enriched")

print("\nFetch outcomes:")
print(df_enriched["site_status"].value_counts(dropna=False).to_string())
print("\nKeyword hits by family (only families this scenario scores):")
for fam, cfg_key in KEYWORD_FAMILIES.items():
    if CONFIG["ICP"].get(cfg_key):
        n = int((df_enriched[f"{fam}_match_count"] > 0).sum())
        print(f"   {fam:11} {n}/{len(df_enriched)} sites matched")
display(df_enriched[["company_name", "site_status", "site_language",
                     "sector_matches", "role_matches", "credential_matches"]])


---
## Cell 7 — Stage 4: Verify

This is the stage that justifies charging for "verified" leads, so be precise
about what it does and does not prove.

**It checks:** syntax is valid, the domain is not a free/disposable provider,
and the domain publishes MX records — i.e. it can receive mail at all.

**It does not check:** whether that specific mailbox exists. Real per-mailbox
verification needs SMTP probing, which gets your IP blacklisted, or a paid API
(NeverBounce, ZeroBounce). Say `mx_valid` to clients, not "100% deliverable" —
overclaiming here is how lead gen freelancers lose accounts.


In [ ]:
# --- Cell 7: Stage 4 — Verify --------------------------------------------
STRICT_EMAIL_RE = re.compile(r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$")

FREE_PROVIDERS = {
    "gmail.com","yahoo.com","hotmail.com","outlook.com","aol.com","icloud.com",
    "qq.com","163.com","126.com","sina.com","foxmail.com","yandex.com","gmx.com",
}
DISPOSABLE = {
    "mailinator.com","guerrillamail.com","10minutemail.com","tempmail.com",
    "throwawaymail.com","yopmail.com","trashmail.com",
}

_MX_CACHE = {}

def has_mx(domain):
    # True / False / None(lookup failed). Cached per domain.
    if not domain:
        return None
    if domain in _MX_CACHE:
        return _MX_CACHE[domain]
    try:
        answers = dns.resolver.resolve(domain, "MX", lifetime=5.0)
        result = len(answers) > 0
    except (dns.resolver.NXDOMAIN, dns.resolver.NoAnswer):
        result = False
    except Exception:
        result = None                      # timeout / server failure: unknown
    _MX_CACHE[domain] = result
    time.sleep(0.2)
    return result


def verify(df):
    d = df.copy()
    d["email_syntax_ok"] = d["email"].apply(
        lambda e: bool(STRICT_EMAIL_RE.match(e)) if e else False)

    d["email_domain"] = d["email"].apply(
        lambda e: e.split("@")[-1].lower() if e and "@" in e else None)

    d["is_free_provider"]  = d["email_domain"].apply(lambda x: x in FREE_PROVIDERS if x else False)
    d["is_disposable"]     = d["email_domain"].apply(lambda x: x in DISPOSABLE if x else False)

    uniq = sorted({x for x in d["email_domain"].dropna().unique()})
    log.info(f"MX lookup on {len(uniq)} unique domains")
    for dom in tqdm(uniq, desc="MX lookups"):
        has_mx(dom)
    d["mx_valid"] = d["email_domain"].apply(has_mx)

    def verdict(r):
        if not r["email"]:            return "no_email"
        if not r["email_syntax_ok"]:  return "invalid_syntax"
        if r["is_disposable"]:        return "disposable"
        if r["mx_valid"] is False:    return "no_mx"
        if r["mx_valid"] is None:     return "mx_unknown"
        if r["is_free_provider"]:     return "valid_free_provider"
        return "valid_business"

    d["email_verdict"] = d.apply(verdict, axis=1)
    # Only these two verdicts count as verified in the scorer.
    d["email_verified"] = d["email_verdict"].isin(["valid_business", "valid_free_provider"])

    log.info("Verification outcomes:\n" + d["email_verdict"].value_counts().to_string())
    return d


df_verified = verify(df_enriched)
save_checkpoint(df_verified, "04_verified")
display(df_verified[["company_name", "email", "email_verdict", "mx_valid", "email_verified"]])


---
## Cell 8 — Stage 5: Score

One scorer, four scenarios. It reads `CONFIG["WEIGHTS"]` and scores **only the
dimensions weighted above zero** — so a `client` run never mentions
certifications, and a `supplier` run never mentions Mandarin-market edge unless
the weights say to.

Every point carries a written reason. `score_reasons` is what lets a client ask
"why is this an 82?" and get a straight answer. Most freelancers hand over an
unexplained ranking; this is the difference.

`max_possible` is tracked separately from `lead_score`, because two leads can
both score 60 for very different reasons — one that hit most of what was
checkable, one that simply had more unknowns. `confidence` reports the share of
dimensions that had data at all.


In [ ]:
# --- Cell 8: Stage 5 — Score ---------------------------------------------
W   = CONFIG["WEIGHTS"]
ICP = CONFIG["ICP"]

def title_norm(t):
    # "Chief Executive Officer (CEO)" -> "chief executive officer ceo"
    # "C.E.O."                        -> "ceo"    (dots dissolved, not split)
    # "总经理"                          -> "总经理"  (CJK preserved)
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    s = str(t).lower()
    s = re.sub(r"[.\u2019']", "", s)
    s = re.sub(r"[^a-z0-9\s\u4e00-\u9fff\u3400-\u4dbf]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def title_matches(title, keywords):
    for kw in keywords:
        if CJK_RE.search(kw):
            if kw in title:
                return kw
        elif re.search(r"\b" + re.escape(kw) + r"\b", title):
            return kw
    return None


def check_gates(r):
    # Returns a disqualification reason, or None. Gates fire only on positive
    # evidence of a miss — an unknown location is not a disqualification.
    g = CONFIG.get("HARD_GATES", {})
    if g.get("require_service_area"):
        city    = (r.get("city") or "").lower()
        country = (r.get("country") or "").lower()
        if (r.get("locality_match_count") or 0) > 0:
            return None
        if city and any(c in city for c in ICP["target_cities"]):
            return None
        if country and not any(c in country for c in ICP["target_countries"]):
            return f"outside service country ('{country}')"
        if city:
            return f"outside service area ('{city}')"
    return None


def _graded(count, weight, label):
    # Shared shape for keyword dimensions: 2+ hits full, 1 hit half, 0 none.
    if count >= 2:
        return weight, f"+{weight} {count} {label} signals on site"
    if count == 1:
        return weight // 2, f"+{weight // 2} 1 {label} signal on site"
    return 0, f"+0 no {label} signal found"


def score_one(r):
    # Reason-before-score. Only dimensions with weight > 0 are evaluated.
    pts, reasons, known = 0, [], 0
    max_possible = sum(w for w in W.values() if w > 0)

    # --- email verified ---
    if W["email_verified"]:
        v = r["email_verdict"]
        if v == "valid_business":
            pts += W["email_verified"]; known += 1
            reasons.append(f"+{W['email_verified']} business email, valid MX")
        elif v == "valid_free_provider":
            half = W["email_verified"] // 2
            pts += half; known += 1
            reasons.append(f"+{half} valid email but free provider")
        else:
            if v != "mx_unknown":
                known += 1
            reasons.append(f"+0 email not verified ({v})")

    # --- decision maker ---
    if W["decision_maker"]:
        title = title_norm(r.get("job_title"))
        if not title:
            reasons.append("+0 no title on record")
        else:
            known += 1
            if (hit := title_matches(title, ICP["excluded_titles"])):
                reasons.append(f"+0 excluded title ('{hit}')")
            elif (hit := title_matches(title, ICP["decision_maker_titles"])):
                pts += W["decision_maker"]
                reasons.append(f"+{W['decision_maker']} decision-maker title ('{hit}')")
            elif (hit := title_matches(title, ICP["influencer_titles"])):
                half = W["decision_maker"] // 2
                pts += half
                reasons.append(f"+{half} influencer title ('{hit}'), not budget holder")
            else:
                reasons.append("+0 title present but unrecognised")

    # --- company size ---
    if W["company_size_fit"]:
        emp = r.get("employee_count")
        if emp is None or pd.isna(emp):
            reasons.append("+0 employee count unknown")
        else:
            known += 1
            emp = int(emp)
            if ICP["employee_min"] <= emp <= ICP["employee_max"]:
                pts += W["company_size_fit"]
                reasons.append(f"+{W['company_size_fit']} size {emp} within ICP band")
            else:
                reasons.append(f"+0 size {emp} outside band "
                               f"({ICP['employee_min']}-{ICP['employee_max']})")

    # --- sector / role / credential (site evidence) ---
    for dim, fam, label in [("sector_match", "sector", "sector"),
                            ("role_match", "role", "supply-chain role"),
                            ("credential_match", "credential", "credential")]:
        if not W[dim]:
            continue
        n = r.get(f"{fam}_match_count") or 0
        if r.get("site_status") != "ok":
            # Site unreadable. For sector only, fall back to the CRM label at
            # reduced value. One reason per dimension, never two.
            crm = (r.get("industry") or "").lower() if fam == "sector" else ""
            if crm and any(k in crm for k in ICP["sector_keywords"]):
                third = W[dim] // 3
                pts += third
                reasons.append(f"+{third} {label} from CRM label only, site unreadable")
            else:
                reasons.append(f"+0 {label} unknown (site not readable)")
            continue
        known += 1
        p, why = _graded(n, W[dim], label)
        pts += p
        if n:
            why += f" [{r.get(f'{fam}_matches')}]"
        reasons.append(why)

    # --- APAC / Mandarin edge ---
    if W["apac_mandarin_edge"]:
        country = (r.get("country") or "").lower()
        if country or r.get("site_language"):
            known += 1
        if any(m in country for m in ICP["mandarin_markets"]) or r.get("site_language") == "zh":
            pts += W["apac_mandarin_edge"]
            reasons.append(f"+{W['apac_mandarin_edge']} Mandarin-market lead (outreach edge)")
        elif any(c in country for c in ICP["target_countries"]):
            half = W["apac_mandarin_edge"] // 2
            pts += half
            reasons.append(f"+{half} APAC target country")
        else:
            reasons.append("+0 outside target geography")

    # --- local proximity (member scenario) ---
    if W["local_proximity"]:
        city = (r.get("city") or "").lower()
        cities = ICP["target_cities"]
        site_hit = (r.get("locality_match_count") or 0) > 0
        if city or site_hit:
            known += 1
        if city and any(c in city for c in cities):
            pts += W["local_proximity"]
            reasons.append(f"+{W['local_proximity']} in service area ('{city}')")
        elif site_hit:
            half = W["local_proximity"] // 2
            pts += half
            reasons.append(f"+{half} service area mentioned on site, address unconfirmed")
        else:
            reasons.append(f"+0 outside service area ('{city or 'unknown'}')")

    # --- website live ---
    if W["website_live"]:
        known += 1
        if r.get("site_status") == "ok":
            pts += W["website_live"]
            reasons.append(f"+{W['website_live']} website reachable")
        else:
            reasons.append(f"+0 website not reachable ({r.get('site_status')})")

    gate = check_gates(r)
    if gate:
        reasons.append(f"DISQUALIFIED: {gate}")

    n_dims = len(ACTIVE)
    return pts, " | ".join(reasons), max_possible, round(known / n_dims, 2), gate


def tier_of(score):
    t = CONFIG["TIERS"]
    if score >= t["A"]: return "A"
    if score >= t["B"]: return "B"
    if score >= t["C"]: return "C"
    return "D"


def score(df):
    d = df.copy()
    res = d.apply(score_one, axis=1)
    d["lead_score"]    = [x[0] for x in res]
    d["score_reasons"] = [x[1] for x in res]
    d["max_possible"]  = [x[2] for x in res]
    d["confidence"]    = [x[3] for x in res]
    d["disqualify_reason"] = [x[4] for x in res]
    d["disqualified"]  = d["disqualify_reason"].notna()
    d["tier"]          = d["lead_score"].apply(tier_of)
    # A gated-out lead is not a low-scoring lead, it is a non-lead. Tier X.
    d.loc[d["disqualified"], "tier"] = "X"
    d["scenario"]      = CONFIG["scenario"]
    d = d.sort_values(["disqualified", "lead_score", "confidence"],
                      ascending=[True, False, False]).reset_index(drop=True)
    d["rank"] = range(1, len(d) + 1)

    log.info(f"Scenario '{CONFIG['scenario']}' — tier distribution:\n"
             + d["tier"].value_counts().sort_index().to_string())
    if d["disqualified"].any():
        log.warning(f"{int(d['disqualified'].sum())} leads disqualified by hard gate "
                    f"— excluded from delivery")
    live = d[~d["disqualified"]]
    log.info(f"Deliverable {len(live)}/{len(d)} | mean score "
             f"{live['lead_score'].mean() if len(live) else 0:.1f} | "
             f"mean confidence {live['confidence'].mean() if len(live) else 0:.0%}")
    return d


df_scored = score(df_verified)
save_checkpoint(df_scored, "05_scored")
display(df_scored[["rank", "company_name", "lead_score", "tier",
                   "confidence", "email_verdict"]].head(20))

print(f"\nWorked example — scenario '{CONFIG['scenario']}':")
top = df_scored.iloc[0]
print(f"{top['company_name']}: {top['lead_score']}/{top['max_possible']} "
      f"(tier {top['tier']}, confidence {top['confidence']:.0%})")
for reason in top["score_reasons"].split(" | "):
    print("   " + reason)


---
## Cell 9 — Measuring whether the score actually works

Right now the rubric is a **hypothesis**, not a validated model. It is built on
plausible reasoning, and plausible reasoning is often wrong.

This cell does nothing useful until you have outcome data — a `replied` or
`booked` column from a campaign that has actually run. Once you do, it compares
your score against a **named baseline: random ordering**. If your top-decile
reply rate is not clearly above the random rate, the rubric is decoration.

Do not put lift numbers in a proposal before running this. Run 200 leads,
measure, then quote the real figure.


In [ ]:
# --- Cell 9: validation (needs outcome data) ------------------------------
OUTCOME_COL = "replied"     # set to your outcome column once you have one

def validate(df, outcome_col=OUTCOME_COL, k_pct=0.20, n_bootstrap=1000):
    if outcome_col not in df.columns:
        print(f"No '{outcome_col}' column yet — nothing to validate.")
        print("Run a campaign, join the replies back on 'email', then re-run this cell.")
        return None

    d = df.dropna(subset=[outcome_col]).copy()
    d[outcome_col] = d[outcome_col].astype(int)
    if len(d) < 30:
        print(f"Only {len(d)} labelled rows. Too few to conclude anything. Need 100+.")
        return None

    import numpy as np
    k = max(1, int(len(d) * k_pct))
    top_k_rate = d.nlargest(k, "lead_score")[outcome_col].mean()
    overall    = d[outcome_col].mean()

    # Baseline: shuffle the ranking, take the same top-k, repeat.
    rng = np.random.default_rng(42)
    vals = d[outcome_col].values
    random_rates = [rng.permutation(vals)[:k].mean() for _ in range(n_bootstrap)]
    baseline = float(np.mean(random_rates))
    p95      = float(np.percentile(random_rates, 95))

    print(f"Labelled leads:            {len(d)}")
    print(f"Top {int(k_pct*100)}% by score (n={k}):  {top_k_rate:.1%} reply rate")
    print(f"Random baseline:           {baseline:.1%} (95th pct: {p95:.1%})")
    print(f"Overall rate:              {overall:.1%}")
    print(f"Lift vs baseline:          {top_k_rate / baseline:.2f}x" if baseline > 0 else "")
    print()
    if top_k_rate > p95:
        print("Score beats random beyond the 95th percentile. The rubric is doing work.")
    else:
        print("Score is inside random noise. Reweight before quoting any lift figure.")
    return {"top_k_rate": top_k_rate, "baseline": baseline, "n": len(d)}


validate(df_scored)


---
## Cell 10 — Stage 6: Export

Builds the client deliverable: a five-tab XLSX matching the structure you used
on the Allianz file, with a formatted summary tab and frozen headers.

Note the **Scored** tab excludes internal columns. Clients get the ranking and
the reasons, not your intermediate scraping fields.


In [ ]:
# --- Cell 10: Stage 6 — Export -------------------------------------------
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

CLIENT_COLS = ["rank", "tier", "lead_score", "max_possible", "confidence",
               "company_name", "website", "domain",
               "contact_name", "job_title", "email", "email_verdict",
               "country", "city", "employee_count", "industry",
               "sector_matches", "role_matches", "credential_matches",
               "site_language", "linkedin_url", "score_reasons"]

def build_summary(df):
    rows = [
        ("Client",              CONFIG["client_name"]),
        ("Campaign",            CONFIG["campaign_name"]),
        ("Scenario",            f"{CONFIG['scenario']} — {CONFIG['label']}"),
        ("Generated (UTC)",     datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")),
        ("Run ID",              RUN_ID),
        ("", ""),
        ("Raw rows ingested",   len(df_raw)),
        ("After cleaning",      len(df_clean)),
        ("Rows dropped",        len(df_raw) - len(df_clean)),
        ("Deliverable leads",   int((~df["disqualified"]).sum())),
        ("Disqualified by gate", int(df["disqualified"].sum())),
        ("", ""),
        ("Tier A (>=%d)" % CONFIG["TIERS"]["A"], int((df["tier"] == "A").sum())),
        ("Tier B (>=%d)" % CONFIG["TIERS"]["B"], int((df["tier"] == "B").sum())),
        ("Tier C (>=%d)" % CONFIG["TIERS"]["C"], int((df["tier"] == "C").sum())),
        ("Tier D (below)",                        int((df["tier"] == "D").sum())),
        ("", ""),
        ("Business emails, MX valid", int((df["email_verdict"] == "valid_business").sum())),
        ("Free-provider emails",      int((df["email_verdict"] == "valid_free_provider").sum())),
        ("Unverified / no email",     int((~df["email_verified"]).sum())),
        ("", ""),
        ("Websites reachable",        int((df["site_status"] == "ok").sum())),
        ("Chinese-language sites",    int((df["site_language"] == "zh").sum())),
        ("Mean score",                round(float(df["lead_score"].mean()), 1)),
        ("Mean confidence",           f"{df['confidence'].mean():.0%}"),
        ("Scoring dimensions used",   ", ".join(f"{k}({v})" for k, v in
                                                sorted(ACTIVE.items(), key=lambda x: -x[1]))),
        ("", ""),
        ("Verification method", "Syntax + domain MX record. Not per-mailbox SMTP."),
        ("Notes",               CONFIG["run_notes"]),
    ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


def export(df):
    fname = f"{CONFIG['campaign_name']}_leads_{RUN_ID}.xlsx"
    path  = os.path.join(CONFIG["OUTPUT_DIR"], fname)

    deliverable = df[~df["disqualified"]]
    gated       = df[df["disqualified"]]
    client_view = deliverable[[c for c in CLIENT_COLS if c in df.columns]].copy()

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        build_summary(df).to_excel(xw, sheet_name="Summary", index=False)
        client_view.to_excel(xw, sheet_name="Scored Leads", index=False)
        client_view[client_view["tier"].isin(["A", "B"])].to_excel(
            xw, sheet_name="Priority (A-B)", index=False)
        if len(gated):
            gated[[c for c in CLIENT_COLS + ["disqualify_reason"]
                   if c in gated.columns]].to_excel(
                xw, sheet_name="Out of Area", index=False)
        df_dropped.to_excel(xw, sheet_name="Rejected", index=False)
        df_raw.to_excel(xw, sheet_name="Raw Source", index=False)

        head_font = Font(bold=True, color="FFFFFF", size=11)
        head_fill = PatternFill("solid", start_color="2F5597")

        for sheet in xw.book.worksheets:
            for cell in sheet[1]:
                cell.font, cell.fill = head_font, head_fill
                cell.alignment = Alignment(horizontal="left", vertical="center")
            sheet.freeze_panes = "A2"
            for col in sheet.columns:
                letter = get_column_letter(col[0].column)
                width = max((len(str(c.value)) for c in col if c.value), default=10)
                sheet.column_dimensions[letter].width = min(max(width + 2, 12), 55)

    log.info(f"Exported: {path}")
    return path, client_view


output_path, client_view = export(df_scored)

# --- optional: push to Google Sheets ---
if CONFIG["WRITE_TO_SHEET"] and CONFIG["OUTPUT_SHEET_ID"]:
    from google.colab import auth
    import gspread
    from gspread_dataframe import set_with_dataframe
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(CONFIG["OUTPUT_SHEET_ID"])
    for tab, frame in [("Scored Leads", client_view), ("Summary", build_summary(df_scored))]:
        try:
            ws = sh.worksheet(tab)
            ws.clear()
        except Exception:
            ws = sh.add_worksheet(title=tab, rows=len(frame) + 10, cols=len(frame.columns) + 5)
        set_with_dataframe(ws, frame)
    log.info("Pushed to Google Sheet.")

# --- download ---
try:
    from google.colab import files
    files.download(output_path)
except Exception:
    print(f"Not in Colab. File is at: {output_path}")

display(build_summary(df_scored))


---
## Cell 11 — Known failure modes

Written down deliberately. When a client asks what the pipeline misses, having
this list ready is worth more than pretending it misses nothing.

| Failure mode | Effect | Mitigation |
|---|---|---|
| JS-rendered sites | `site_text_chars` near zero, industry score wrongly 0 | Check the low-text rows by hand; Playwright if it becomes common |
| Cloudflare / bot walls | `http_403`, website points lost | Treat 403 as unknown, not absent; verify by hand |
| Catch-all mail domains | `mx_valid` True but mailbox may not exist | Say "MX valid", never "deliverable" |
| MX timeout | `mx_unknown` scores 0, penalising a possibly good lead | Re-run Cell 7; the cache keeps it cheap |
| Parked domains | Site resolves, content is a placeholder | Low `site_text_chars` plus no keyword hits flags these |
| Company rebrand | Old domain 404s, lead looks dead | Search the company name before discarding |
| Multiple contacts, one company | Domain dedupe may drop a better contact | Dedupe key is (domain, email), so distinct people survive |
| Rubric never validated | Confident ranking, no evidence | Cell 9 against a random baseline before quoting lift |
| Sector/role confusion | A manufacturer and its distributor share sector words | Role keywords are scored separately; check `role_matches` |
| Self-declared credentials | Site says "ISO 9001", nobody verified it | `credential_match` measures *claims*. Ask for the certificate before ordering |
| Proximity from city string | "Bandung" in a CRM field may be a billing address | `local_proximity` gives half points for site-only evidence. Confirm before a site visit |
| Member scenario scope | Scores organisations, not individual walk-ins | Individual prospects need a different source entirely |

**Two things to keep straight for client work:**

1. `RESPECT_ROBOTS` is `True` by default. Leave it. A blacklisted client domain
   costs far more than the leads you would gain.
2. For EU or UK contacts, GDPR applies to B2B personal data. Legitimate interest
   can cover B2B outreach, but you need a lawful basis and a working opt-out.
   The `target_countries` default is APAC-only, which sidesteps this — widen it
   deliberately, not accidentally.


---
## Next steps

1. Run top to bottom on `INPUT_MODE = "sample"` and confirm the export downloads.
2. Switch to `csv_upload` with 25 real rows, `ENRICH_LIMIT = 25`. Read every row
   of the output by hand and check the score reasons match your judgment.
3. Adjust `WEIGHTS` where they disagree with you. This is where the rubric gets good.
4. Remove `ENRICH_LIMIT`, run the full list.
5. After the first campaign, join reply data back and run Cell 9.

**Save your work:** File → Save a copy in GitHub, into your
`claude-agent-suite` repo. Colab wipes `/content` on disconnect; the notebook
itself is only safe once it is in Drive or GitHub.
